# Кластеризация: основы, метрики и K-Means

> Модуль **Unsupervised Learning**, ноутбук 2. Это **справочный** ноутбук: тема разобрана
> подробно и чуть академичнее, чем предыдущий. Не нужно запоминать всё сразу — сюда стоит
> возвращаться, когда понадобится конкретная формула или метод.

### План занятия
1. **Постановка задачи** — что такое кластеризация, чем отличается от классификации, какие
   бывают виды, почему это «некорректно поставленная» задача.
2. **Расстояния и масштабирование** — на чём держится понятие «похожести».
3. **Метрики качества** — внутренние (без меток) и внешние (с метками): inertia, silhouette,
   Davies–Bouldin, Calinski–Harabasz, ARI, NMI, V-measure.
4. **K-Means** — функционал, алгоритм Ллойда, вывод формулы центроида, инициализация
   k-means++, сходимость и предположения, где метод ломается.
5. **Выбор числа кластеров $k$** — метод локтя, силуэт, и другие.
6. **Практический пример** — восстановление трёх дикторов по их эмбеддингам.

В конце — раздел **вопросов и ответов** с типичными граблями.

## 1. Постановка задачи

В **обучении с учителем** (классификация, регрессия) у каждого объекта есть правильный ответ
$y_i$, и модель учится его воспроизводить. В **кластеризации** ответов нет: дана только
матрица объектов–признаков

$$ X = \{x_1, x_2, \dots, x_n\}, \qquad x_i \in \mathbb{R}^d, $$

и требуется **разбить объекты на группы (кластеры)** так, чтобы внутри группы объекты были
**похожи**, а объекты из разных групп — **различны**. Меток классов нам никто не даёт — мы
ищем структуру «изнутри» самих данных.

> **Определение.** *Кластеризация* — это разбиение множества объектов на подмножества
> (кластеры) на основании меры их сходства, без использования заранее известных меток.

### Чем кластеризация сложнее классификации

- **Нет «правильного ответа».** Одни и те же данные можно осмысленно разбить по-разному
  (людей — по возрасту или по доходу). Что считать «хорошим» разбиением, зависит от **меры
  сходства** и от задачи. Поэтому говорят, что кластеризация — *некорректно поставленная*
  (ill-posed) задача.
- **Нечего «проверить на отложенной выборке»** в привычном смысле — оценивать качество
  приходится косвенно (об этом — раздел 3).
- **Число групп $k$ часто неизвестно** и само является предметом подбора.

### Виды кластеризации

| Признак | Варианты |
|---|---|
| **жёсткость** | *жёсткая* (объект ровно в одном кластере) vs *мягкая* (вероятности принадлежности — глава про EM) |
| **структура** | *плоская* (K-Means) vs *иерархическая* (дерево вложенных кластеров) |
| **принцип** | *по центрам* (K-Means), *по плотности* (DBSCAN/HDBSCAN), *вероятностная* (GMM/EM) |
| **число кластеров** | задаётся заранее (K-Means) vs определяется автоматически (DBSCAN, иерархическая) |

Этот ноутбук — про самый распространённый метод, **K-Means** (жёсткий, плоский, по центрам).
Остальные семейства — в следующих ноутбуках.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_completeness_v_measure)

try:
    from ipywidgets import interact, IntSlider
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

np.random.seed(0)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})
PALETTE = ["#9b3b46", "#2f6f4f", "#2d5d8a", "#9a6b00", "#5b4b8a",
           "#3a7d7b", "#a0522d", "#557a2f", "#8a3b6b", "#3b6ea0"]

# A clean toy dataset we will reuse for K-Means
X_blob, y_blob = make_blobs(n_samples=600, centers=4, cluster_std=0.85, random_state=42)
print("toy data:", X_blob.shape, "| истинных групп:", len(np.unique(y_blob)))
print("Виджеты:", "да" if HAVE_WIDGETS else "нет (графики статичные)")

## 2. Похожесть = расстояние. Зачем масштабировать признаки

«Похожи» формализуют через **расстояние** между объектами: чем меньше расстояние, тем больше
сходство. Самое частое — **евклидово**:

$$ \rho(x, z) = \sqrt{\sum_{j=1}^{d} (x_j - z_j)^2}. $$

Бывают и другие: **манхэттенское** $\sum_j |x_j - z_j|$ (устойчивее к выбросам),
**косинусное** $1 - \frac{\langle x, z\rangle}{\|x\|\,\|z\|}$ (для текстов и эмбеддингов, когда
важно направление, а не длина).

> **Критично: масштаб признаков.** Евклидово расстояние складывает вклады всех признаков.
> Если один признак измеряется в тысячах (зарплата), а другой — в единицах (возраст), то
> расстояние будет определяться почти только зарплатой, а возраст «утонет». Поэтому перед
> кластеризацией признаки обычно приводят к одному масштабу (`StandardScaler`: вычесть среднее,
> поделить на стандартное отклонение). Покажем, как масштаб ломает разбиение:

In [ ]:
# Feature 0 carries the real separation; feature 1 is pure noise but on a huge scale.
rng = np.random.default_rng(0)
n = 150
x0 = np.r_[rng.normal(-2.0, 0.6, n), rng.normal(2.0, 0.6, n)]   # informative
x1 = rng.normal(0.0, 1.0, 2 * n)                                # noise
true = np.r_[np.zeros(n), np.ones(n)].astype(int)
X_raw = np.c_[x0, x1].copy()
X_raw[:, 1] *= 40                                               # blow up the noise scale

lab_raw = KMeans(2, n_init=10, random_state=0).fit_predict(X_raw)
lab_scaled = KMeans(2, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(X_raw))
print("ARI без масштабирования:", round(adjusted_rand_score(true, lab_raw), 3),
      "| со StandardScaler:", round(adjusted_rand_score(true, lab_scaled), 3))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.scatter(X_raw[:, 0], X_raw[:, 1], c=[PALETTE[i] for i in lab_raw], s=12)
a1.set_title("Без масштабирования: делит по шуму (неверно)")
a1.set_xlabel("информативный признак"); a1.set_ylabel("шум × 40")
a2.scatter(x0, x1, c=[PALETTE[i] for i in lab_scaled], s=12)
a2.set_title("Со StandardScaler: делит правильно")
a2.set_xlabel("информативный признак"); a2.set_ylabel("шум")
plt.show()

## 3. Метрики качества кластеризации

Как понять, что разбиение «хорошее»? Метрики делятся на две группы.

### 3.1. Внутренние (не требуют меток)

Опираются только на геометрию: кластеры должны быть **плотными** внутри и **далёкими** друг
от друга.

**Inertia (WCSS, within-cluster sum of squares)** — суммарный разброс внутри кластеров:

$$ \mathrm{WCSS} = \sum_{c=1}^{k} \sum_{x_i \in C_c} \|x_i - \mu_c\|^2, $$

где $\mu_c$ — центр кластера $C_c$. Чем меньше, тем плотнее кластеры. **Но**: WCSS всегда
убывает с ростом $k$ (при $k=n$ она равна нулю), поэтому сравнивать по ней разные $k$ напрямую
нельзя — нужен «локоть» (раздел 5).

**Силуэт (silhouette).** Для каждого объекта $i$:
- $a(i)$ — среднее расстояние до объектов **своего** кластера (насколько он «свой»);
- $b(i)$ — среднее расстояние до объектов **ближайшего чужого** кластера;

$$ s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}} \in [-1, 1]. $$

Близко к $+1$ — объект уверенно в своём кластере; около $0$ — на границе; отрицательно —
вероятно, попал не туда. **Силуэтный коэффициент** разбиения — среднее $s(i)$ по всем
объектам. В отличие от WCSS, его *можно* сравнивать между разными $k$.

**Davies–Bouldin** (меньше — лучше) и **Calinski–Harabasz** (больше — лучше) — ещё две
популярные внутренние метрики, тоже основанные на отношении «разброс внутри / расстояние
между» кластерами.

In [ ]:
# Silhouette "knife" plot: each cluster is a band of sorted per-point silhouettes
def silhouette_plot(k=4):
    labels = KMeans(k, n_init=10, random_state=0).fit_predict(X_blob)
    sil = silhouette_samples(X_blob, labels)
    score = sil.mean()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    y_low = 0
    for c in range(k):
        vals = np.sort(sil[labels == c])
        a1.fill_betweenx(np.arange(y_low, y_low + len(vals)), 0, vals,
                         color=PALETTE[c], alpha=0.8)
        a1.text(-0.05, y_low + len(vals) / 2, str(c), va="center")
        y_low += len(vals) + 10
    a1.axvline(score, color="#9b3b46", ls="--", label=f"средний силуэт = {score:.3f}")
    a1.set_xlabel("s(i)"); a1.set_yticks([]); a1.legend(loc="lower right")
    a1.set_title(f"Силуэтный профиль (k={k})")
    a2.scatter(X_blob[:, 0], X_blob[:, 1], c=[PALETTE[i] for i in labels], s=12)
    a2.set_title("Разбиение"); a2.set_xlabel("x1"); a2.set_ylabel("x2")
    plt.show()


if HAVE_WIDGETS:
    interact(silhouette_plot, k=IntSlider(min=2, max=8, step=1, value=4))
else:
    silhouette_plot(4)

### 3.2. Внешние (когда метки всё-таки есть)

Иногда истинные классы известны (как в нашем датасете дикторов) — тогда можно честно сравнить
найденное разбиение с «правдой». Важно: метрики не должны зависеть от того, какими *номерами*
названы кластеры (кластер «0» может соответствовать классу «2»).

- **ARI (Adjusted Rand Index)** — доля согласованных пар объектов («вместе/раздельно»),
  скорректированная на случайность. $1$ — идеальное совпадение, $0$ — на уровне случайного.
- **NMI (Normalized Mutual Information)** — сколько информации о классе несёт кластер,
  нормировано в $[0, 1]$.
- **Однородность / полнота / V-мера** (homogeneity / completeness / V-measure):
  *однородность* — в каждом кластере объекты одного класса; *полнота* — каждый класс целиком
  в одном кластере; *V-мера* — их гармоническое среднее.

In [ ]:
# External metrics on a labelled toy: compare a KMeans partition to true blob labels
labels4 = KMeans(4, n_init=10, random_state=0).fit_predict(X_blob)
h, c, v = homogeneity_completeness_v_measure(y_blob, labels4)
print("ARI :", round(adjusted_rand_score(y_blob, labels4), 3))
print("NMI :", round(normalized_mutual_info_score(y_blob, labels4), 3))
print(f"однородность={h:.3f}  полнота={c:.3f}  V-мера={v:.3f}")

# Label permutation must NOT change the metric:
permuted = np.choose(labels4, [2, 0, 3, 1])  # rename clusters
print("ARI после переименования кластеров:", round(adjusted_rand_score(y_blob, permuted), 3),
      "(не изменился — и это правильно)")

## 4. K-Means

### 4.1. Что оптимизирует

K-Means ищет $k$ центров (центроидов) $\mu_1, \dots, \mu_k$ и приписывает каждый объект к
ближайшему центру так, чтобы минимизировать суммарный внутрикластерный разброс — ту самую
**WCSS**:

$$ J(\{C_c\}, \{\mu_c\}) = \sum_{c=1}^{k} \sum_{x_i \in C_c} \|x_i - \mu_c\|^2 \;\to\; \min. $$

Точно минимизировать $J$ по всем разбиениям — NP-трудная задача. Поэтому используют простой и
эффективный приближённый алгоритм — **алгоритм Ллойда**.

### 4.2. Алгоритм Ллойда

Две операции, повторяемые по очереди до сходимости:

1. **Шаг назначения (Assignment).** При фиксированных центрах отнести каждый объект к
   ближайшему центру: $\;C_c = \{x_i : \|x_i - \mu_c\| \le \|x_i - \mu_{c'}\|\ \forall c'\}$.
2. **Шаг обновления (Update).** При фиксированных назначениях пересчитать центр каждого
   кластера как **среднее** его объектов: $\;\mu_c = \frac{1}{|C_c|}\sum_{x_i \in C_c} x_i$.

<details>
<summary><b>Для любознательных:</b> почему центр — это именно среднее</summary>

На шаге обновления назначения фиксированы, и $J$ распадается на независимые слагаемые по
кластерам. Для одного кластера нужно минимизировать $\sum_{x_i \in C_c}\|x_i - \mu\|^2$ по
$\mu$. Возьмём производную по $\mu$ и приравняем нулю:

$$ \frac{\partial}{\partial \mu}\sum_{x_i \in C_c}\|x_i - \mu\|^2
   = -2\sum_{x_i \in C_c}(x_i - \mu) = 0
   \;\Longrightarrow\; \mu = \frac{1}{|C_c|}\sum_{x_i \in C_c} x_i. $$

То есть лучший центр по квадратичному критерию — среднее (центр масс). Это та же «лучшая
константа», что и в листе регрессионного дерева: для суммы квадратов оптимум — среднее.
</details>

**Сходимость.** Каждый из двух шагов **не увеличивает** $J$ (назначение к ближайшему центру не
может ухудшить сумму; среднее — оптимальный центр). Значит $J$ монотонно убывает и алгоритм
сходится — но к **локальному** минимуму, зависящему от начальных центров. Посмотрим итерации
вживую:

In [ ]:
# Manual Lloyd iterations so we can watch each step
def kmeans_history(X, k, seed, n_iter=25):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)].astype(float)  # random init
    hist = []
    for _ in range(n_iter):
        dist = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        labels = dist.argmin(1)
        inertia = (dist[np.arange(len(X)), labels] ** 2).sum()
        hist.append((centers.copy(), labels.copy(), inertia))
        new = np.array([X[labels == c].mean(0) if np.any(labels == c) else centers[c]
                        for c in range(k)])
        if np.allclose(new, centers):
            break
        centers = new
    return hist


HIST = kmeans_history(X_blob, k=4, seed=3)


def show_lloyd(iteration=0):
    centers, labels, inertia = HIST[min(iteration, len(HIST) - 1)]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(X_blob[:, 0], X_blob[:, 1], c=[PALETTE[i] for i in labels], s=14, alpha=0.6)
    ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=220, edgecolor="white")
    ax.set_title(f"Итерация {iteration} из {len(HIST)-1}  ·  WCSS = {inertia:.0f}")
    plt.show()


if HAVE_WIDGETS:
    interact(show_lloyd, iteration=IntSlider(min=0, max=len(HIST) - 1, step=1, value=0))
else:
    show_lloyd(len(HIST) - 1)

### 4.3. Инициализация: k-means++

Случайные начальные центры могут привести к плохому локальному минимуму (два центра в одной
группе, ни одного — в другой). Два стандартных лекарства:

- **k-means++** — умная инициализация: первый центр случаен, а каждый следующий выбирается с
  вероятностью, пропорциональной **квадрату расстояния** до ближайшего уже выбранного центра.
  Центры «расталкиваются» по разным областям. Это поведение по умолчанию в `sklearn`.
- **`n_init`** — запустить алгоритм несколько раз из разных начал и взять лучший по WCSS
  результат.

### 4.4. Предположения и где K-Means ломается

K-Means неявно считает, что кластеры — **выпуклые, примерно сферические, сопоставимого размера
и плотности**. Это прямое следствие критерия (евклидовы шары вокруг центров). Когда форма иная,
метод ошибается:

In [ ]:
# Three classic failure modes for K-Means
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (1) anisotropic (stretched) clusters
Xa, ya = make_blobs(n_samples=600, centers=3, cluster_std=0.7, random_state=2)
Xa = Xa @ np.array([[0.6, -0.6], [-0.4, 0.8]])
la = KMeans(3, n_init=10, random_state=0).fit_predict(Xa)
axes[0].scatter(Xa[:, 0], Xa[:, 1], c=[PALETTE[i] for i in la], s=10)
axes[0].set_title("Вытянутые (anisotropic):\nрежет поперёк формы")

# (2) unequal variance
Xv, yv = make_blobs(n_samples=600, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=2)
lv = KMeans(3, n_init=10, random_state=0).fit_predict(Xv)
axes[1].scatter(Xv[:, 0], Xv[:, 1], c=[PALETTE[i] for i in lv], s=10)
axes[1].set_title("Разная дисперсия:\nразмывает крупный кластер")

# (3) non-convex (moons)
Xm, ym = make_moons(n_samples=600, noise=0.06, random_state=0)
lm = KMeans(2, n_init=10, random_state=0).fit_predict(Xm)
axes[2].scatter(Xm[:, 0], Xm[:, 1], c=[PALETTE[i] for i in lm], s=10)
axes[2].set_title("Полумесяцы (non-convex):\nне ловит изогнутую форму")
plt.show()
print("Для таких форм лучше подходят плотностные (DBSCAN/HDBSCAN) и др. — следующие ноутбуки.")

## 5. Как выбрать число кластеров $k$

$k$ — главный гиперпараметр K-Means, и обычно он неизвестен. Подходы:

- **Метод локтя (elbow).** Строим WCSS как функцию $k$. Она всегда убывает, но в «правильном»
  $k$ убывание резко замедляется — образуется «локоть». Его и берут.
- **Силуэт.** Строим средний силуэт как функцию $k$ и берём $k$ с **максимумом** (силуэт, в
  отличие от WCSS, не обязан расти с $k$).
- **Calinski–Harabasz / Davies–Bouldin** — ещё два индекса для перебора по $k$.
- **Gap statistic** — сравнивает WCSS с ожидаемой на случайных данных (упоминаем для полноты).

На практике смотрят сразу на несколько и учитывают смысл задачи. Подвигайте $k$:

In [ ]:
# Elbow (WCSS) and silhouette as functions of k
ks = list(range(2, 11))
wcss = [KMeans(k, n_init=10, random_state=0).fit(X_blob).inertia_ for k in ks]
sils = [silhouette_score(X_blob, KMeans(k, n_init=10, random_state=0).fit_predict(X_blob)) for k in ks]


def choose_k(k=4):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(ks, wcss, "o-", color="#2d5d8a"); a1.axvline(k, color="#9b3b46", ls="--")
    a1.set_title("Метод локтя (WCSS)"); a1.set_xlabel("k"); a1.set_ylabel("WCSS")
    a2.plot(ks, sils, "o-", color="#2f6f4f"); a2.axvline(k, color="#9b3b46", ls="--")
    a2.set_title("Средний силуэт"); a2.set_xlabel("k"); a2.set_ylabel("silhouette")
    best = ks[int(np.argmax(sils))]
    plt.show()
    print(f"k={k}: WCSS={wcss[k-2]:.0f}, силуэт={sils[k-2]:.3f}  |  максимум силуэта при k={best}")


if HAVE_WIDGETS:
    interact(choose_k, k=IntSlider(min=2, max=10, step=1, value=4))
else:
    choose_k(4)

> **Важно.** «Локоть» бывает размытым, а разные метрики могут указывать на разное $k$ — это
> нормально для некорректно поставленной задачи. Финальное решение часто диктуется
> **смыслом**: сколько групп осмысленно в вашей предметной области.

### Когда брать K-Means (и когда нет)

- ✅ Кластеры компактные и примерно сферические; данных много (метод быстрый,
  масштабируется); число групп известно или подбирается.
- ⛔ Кластеры вытянутые, разной плотности, сложной формы; много выбросов (центроид
  чувствителен к ним); $k$ принципиально неизвестно — тогда смотрите в сторону плотностных,
  иерархических или вероятностных методов.

**Применения:** сегментация клиентов, сжатие цветов изображения (квантизация палитры),
группировка документов/эмбеддингов, предобработка признаков.

## 6. Практический пример: дикторы по эмбеддингам

Применим K-Means к реальным **128-мерным WavLM-эмбеддингам** из датасета
[`aiacademy-kg/audio-3-speaker-dataset`](https://huggingface.co/datasets/aiacademy-kg/audio-3-speaker-dataset).
Истинные метки (кто говорит) у нас есть — но K-Means их **не видит**; мы используем их только
чтобы проверить качество по внешним метрикам.

In [ ]:
def load_speaker_embeddings(candidates=("speaker_emb.npz", "data/speaker_emb.npz")):
    # Local cache first; otherwise pull ONLY embedding/label columns from HuggingFace
    # (skips the 2.65 GB audio), then cache. One-time remote step ~1-2 min, then instant.
    import os
    for c in candidates:
        if os.path.exists(c):
            d = np.load(c, allow_pickle=True)
            return d["X"], d["y"], d["text"]
    from huggingface_hub import HfApi
    import pyarrow.parquet as pq
    import fsspec
    repo = "aiacademy-kg/audio-3-speaker-dataset"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    fs = fsspec.filesystem("hf")
    E, Y, T = [], [], []
    for f in files:
        pf = pq.ParquetFile(fs.open(f"datasets/{repo}/{f}"))
        d = pf.read(columns=["wavlm_embedding", "speaker_id", "text"]).to_pandas()
        E.append(np.array(d["wavlm_embedding"].tolist(), dtype=np.float32))
        Y += d["speaker_id"].tolist(); T += d["text"].tolist()
    X, y, text = np.vstack(E), np.array(Y), np.array(T, dtype=object)
    np.savez_compressed("speaker_emb.npz", X=X, y=y, text=text)
    return X, y, text


X_emb, y_emb, _ = load_speaker_embeddings()
Xe = StandardScaler().fit_transform(X_emb)
Pe = PCA(n_components=10, random_state=0).fit_transform(Xe)   # denoise + speed up
pred = KMeans(3, n_init=10, random_state=0).fit_predict(Pe)

print("ARI :", round(adjusted_rand_score(y_emb, pred), 3))
print("NMI :", round(normalized_mutual_info_score(y_emb, pred), 3))
print("\nКто в каком кластере (строки — дикторы, столбцы — кластеры K-Means):")
print(pd.crosstab(y_emb, pred))

In [ ]:
# Visual check in 2D PCA: predicted clusters vs true speakers
P2 = PCA(n_components=2, random_state=0).fit_transform(Xe)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.scatter(P2[:, 0], P2[:, 1], c=[PALETTE[i] for i in pred], s=8, alpha=0.5)
a1.set_title("Кластеры K-Means")
spk = {s: i for i, s in enumerate(np.unique(y_emb))}
a2.scatter(P2[:, 0], P2[:, 1], c=[PALETTE[spk[s]] for s in y_emb], s=8, alpha=0.5)
a2.set_title("Истинные дикторы")
for s, i in spk.items():
    a2.scatter([], [], color=PALETTE[i], label=s)
a2.legend(title="диктор")
plt.show()

## 7. Выводы

- **Кластеризация** ищет структуру без меток; задача некорректно поставлена — ответ зависит от
  меры сходства и числа групп.
- **Масштабируйте признаки** перед расчётом расстояний, иначе крупномасштабный признак
  заглушит остальные.
- **Метрики:** внутренние (WCSS, силуэт, DB, CH) — без меток; внешние (ARI, NMI, V-мера) —
  когда метки известны. WCSS нельзя сравнивать между $k$ напрямую, силуэт — можно.
- **K-Means** минимизирует WCSS алгоритмом Ллойда (назначение ↔ обновление-среднее); сходится
  к локальному минимуму, поэтому нужны k-means++ и `n_init`.
- K-Means любит **сферические** кластеры сопоставимого размера; на вытянутых, разноплотных и
  изогнутых формах ошибается.
- **Число $k$** подбирают локтём/силуэтом и здравым смыслом.
- На реальных эмбеддингах K-Means **точно восстановил трёх дикторов** (ARI ≈ 1).

**Дальше:** мягкая, вероятностная кластеризация — **смеси гауссиан и алгоритм EM**, где у нас
появятся не жёсткие метки, а вероятности принадлежности.

## Вопросы и ответы

**1) K-Means каждый раз даёт разный результат — это нормально?**
Да, из-за случайной инициализации он сходится к разным локальным минимумам. Лекарство —
k-means++ и `n_init` (sklearn по умолчанию берёт лучший из нескольких запусков). Для полной
воспроизводимости фиксируйте `random_state`.

**2) Можно ли по WCSS выбрать $k$, взяв минимум?**
Нет: WCSS монотонно убывает с ростом $k$ и при $k=n$ равна нулю. По ней ищут «локоть», а для
прямого сравнения разных $k$ берут силуэт или CH/DB.

**3) Чем силуэт лучше WCSS?**
Силуэт учитывает не только плотность внутри кластера ($a$), но и удалённость от соседнего
($b$), и нормирован в $[-1,1]$, поэтому его можно сравнивать между разными $k$ и датасетами.

**4) K-Means плохо разбил данные — это всегда вина $k$?**
Не обязательно. Возможно, форма кластеров не сферическая (см. раздел 4.4) или признаки не
масштабированы. Сначала проверьте масштабирование и форму, а уже потом — число $k$.

**5) Зачем сначала PCA, а потом K-Means (как в примере с дикторами)?**
PCA убирает шумные направления и снижает размерность — расстояния становятся
«контрастнее», а алгоритм работает быстрее. Это частый и полезный приём для эмбеддингов.

**6) K-Means относит выброс к кластеру — что делать?**
K-Means не умеет говорить «это шум» — каждый объект обязан попасть в какой-то кластер, а
выбросы ещё и тянут центроид на себя. Если важно отделять шум — это плотностные методы
(DBSCAN/HDBSCAN).